# Schema, Service, Exports, and Bundles
This notebook starts from one shipped sample case, then contrasts the protected main-service boundary with unsupported file exports and serialized `PinchWorkspace` views used by repository app and dashboard layers.

**Support notice:** Only the `from OpenPinch.main import pinch_analysis_service` call is compatibility protected. The other owner-module examples in this advanced notebook are unsupported internals.


## Case context
The goal here is not a new thermal story. It is to show how one real packaged JSON case moves through the unsupported file-based `PinchProblem` workflow, the protected mapping-to-`pinch_analysis_service(...)` boundary, and an internal serialized workspace shape without switching examples.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from OpenPinch.application.problem import PinchProblem
from OpenPinch.application.workspace import PinchWorkspace
from OpenPinch.main import pinch_analysis_service
from OpenPinch.contracts.input import TargetInput
from OpenPinch.resources import copy_sample_case


In [ ]:
working_dir = TemporaryDirectory()
local_case_path = copy_sample_case(
    "basic_pinch.json",
    Path(working_dir.name) / "basic_pinch.json",
)

local_case_path


## Local-file `PinchProblem` workflow
This internal coordinator is useful when you already have a JSON file on disk and accept unsupported validation, targeting, graphing, and export APIs.


In [ ]:
problem = PinchProblem(local_case_path, project_name="basic_pinch_local")
validated = problem.validate()
problem.target()

{
    "stream_count": len(validated.streams),
    "utility_count": len(validated.utilities or []),
    "summary_rows": len(problem.summary_frame()),
}


In [ ]:
problem.summary_frame()


## Typed boundary with `TargetInput` and `pinch_analysis_service(...)`
Use the schema model when you want explicit validation at the request boundary before running the analysis service.


In [ ]:
target_input = TargetInput.model_validate_json(
    local_case_path.read_text(encoding="utf-8")
)
service_result = pinch_analysis_service(target_input, project_name="basic_pinch_service")

pd.DataFrame(
    [
        {
            "workflow": "PinchProblem",
            "Qh": problem.summary_frame().iloc[0]["Hot Utility Target"],
            "Qc": problem.summary_frame().iloc[0]["Cold Utility Target"],
            "Qr": problem.summary_frame().iloc[0]["Heat Recovery"],
        },
        {
            "workflow": "pinch_analysis_service",
            "Qh": service_result.targets[0].Qh,
            "Qc": service_result.targets[0].Qc,
            "Qr": service_result.targets[0].Qr,
        },
    ]
)


## Export Excel and graph artifacts
Use `TemporaryDirectory` for teaching notebooks so the workflow is reproducible and does not leave stale artifacts in the working tree.


In [ ]:
artifact_dir = TemporaryDirectory()
excel_path = problem.export_excel(Path(artifact_dir.name) / "excel")
graph_paths = problem.plot.export(Path(artifact_dir.name) / "graphs", graph_type="gcc")

{
    "excel_path": str(excel_path),
    "graph_files": [path.name for path in graph_paths],
}


## Serialized `PinchWorkspace` views
The workspace surface is useful when a caller needs editable input views, validation reports, solved variant views, and deterministic comparison views instead of only live `PinchProblem` objects.


In [ ]:
workspace = PinchWorkspace(source=local_case_path, project_name="basic_workspace")
input_view = workspace.input_view("baseline")
validation = workspace.validate_variant("baseline")

{
    "input_view_sections": {
        "zones": len(input_view.zones),
        "streams": len(input_view.streams),
        "utilities": len(input_view.utilities),
    },
    "first_stream_path": input_view.streams[0].path,
    "validation_valid": validation.valid,
    "option_keys": sorted(input_view.options),
}


In [ ]:
workspace.set_variant_input(
    "tight_dt",
    {"options": {"THERMAL_DT_CONT": 8.0}},
    base="baseline",
)

baseline_view = workspace.solve_variant("baseline")
tight_dt_view = workspace.solve_variant("tight_dt")

pd.DataFrame(
    [
        {
            "variant": baseline_view.variant_name,
            "status": baseline_view.status,
            "summary_cards": len(baseline_view.summary_cards),
            "graph_catalog_size": len(baseline_view.graph_catalog),
            "problem_tables": len(baseline_view.problem_tables),
        },
        {
            "variant": tight_dt_view.variant_name,
            "status": tight_dt_view.status,
            "summary_cards": len(tight_dt_view.summary_cards),
            "graph_catalog_size": len(tight_dt_view.graph_catalog),
            "problem_tables": len(tight_dt_view.problem_tables),
        },
    ]
)


In [ ]:
comparison_view = workspace.compare_variants(["baseline", "tight_dt"])
pd.DataFrame([delta.model_dump() for delta in comparison_view.metric_deltas]).head(8)


In [ ]:
bundle_path = Path(artifact_dir.name) / "workspace_bundle.json"
workspace.save_bundle(bundle_path)
reloaded = PinchWorkspace.load_bundle(bundle_path)

{
    "bundle_path": str(bundle_path),
    "reloaded_variants": reloaded.list_variants(),
    "reloaded_baseline_status": reloaded.solve_variant("baseline").status,
}
